## Suhas Koheda
## 23BAI1148

## Task A
Represent the MDP with states (taxi location, passenger availability),actions, transition probabilities, and rewards.

In [1]:
import numpy as np

zones=["A","B","C","D","E","F","G"]
num_states=len(zones)

passenger_prob = {
    "A": 0.8,  
    "B": 0.6,
    "C": 0.5,
    "D": 0.2,  
    "E": 0.4,
    "F": 0.7,
    "G": 0.3
}
neighbors = {
    "A": ["B", "C"],
    "B": ["A", "D", "E"],
    "C": ["A", "F"],
    "D": ["B", "G"],
    "E": ["B", "F"],
    "F": ["C", "E", "G"],
    "G": ["D", "F"]
}
actions={}
for zone in zones:
    actions[zone]=neighbors[zone]+[zone]
print("Available Actions:\n")
for z in zones:
    print(z,"->",actions[z])

Available Actions:

A -> ['B', 'C', 'A']
B -> ['A', 'D', 'E', 'B']
C -> ['A', 'F', 'C']
D -> ['B', 'G', 'D']
E -> ['B', 'F', 'E']
F -> ['C', 'E', 'G', 'F']
G -> ['D', 'F', 'G']


In [4]:
def reward(current_zone,next_zone):
    p=passenger_prob[next_zone]
    expected_reward=(p*20 + (1-p)*(-5))
    if current_zone!=next_zone:
        expected_reward-=1
    return expected_reward
def transition(current_zone, intended_zone):
    return {
        intended_zone: 0.7,
        current_zone: 0.3
    }
print("\nMDP State,Action,Rewards\n")

for zone in zones:

    print("Current Zone:", zone)

    for act in actions[zone]:

        print(" Action ->", act)

        trans = transition(zone, act)

        for nxt, prob in trans.items():

            r = reward(zone, nxt)

            print(f"    {nxt:2}  Prob={prob:.1f}  Reward={r:.2f}")

    print()


MDP State,Action,Rewards

Current Zone: A
 Action -> B
    B   Prob=0.7  Reward=9.00
    A   Prob=0.3  Reward=15.00
 Action -> C
    C   Prob=0.7  Reward=6.50
    A   Prob=0.3  Reward=15.00
 Action -> A
    A   Prob=0.3  Reward=15.00

Current Zone: B
 Action -> A
    A   Prob=0.7  Reward=14.00
    B   Prob=0.3  Reward=10.00
 Action -> D
    D   Prob=0.7  Reward=-1.00
    B   Prob=0.3  Reward=10.00
 Action -> E
    E   Prob=0.7  Reward=4.00
    B   Prob=0.3  Reward=10.00
 Action -> B
    B   Prob=0.3  Reward=10.00

Current Zone: C
 Action -> A
    A   Prob=0.7  Reward=14.00
    C   Prob=0.3  Reward=7.50
 Action -> F
    F   Prob=0.7  Reward=11.50
    C   Prob=0.3  Reward=7.50
 Action -> C
    C   Prob=0.3  Reward=7.50

Current Zone: D
 Action -> B
    B   Prob=0.7  Reward=9.00
    D   Prob=0.3  Reward=0.00
 Action -> G
    G   Prob=0.7  Reward=1.50
    D   Prob=0.3  Reward=0.00
 Action -> D
    D   Prob=0.3  Reward=0.00

Current Zone: E
 Action -> B
    B   Prob=0.7  Reward=9.00
    E 

## Task B
Implement policy evaluation to estimate the value of each state undera given policy.


In [12]:
policy={}
for zone in zones:
    policy[zone]=actions[zone][0]
print("Initial Policy")
for zone in zones:
    print(f"{zone}->{policy[zone]}")

def policy_evaluation(policy,gamma=0.0,theta=1e-6):
    V={zone: 0 for zone in zones}
    while True:
        delta=0
        for state in zones:
            old_value=V[state]
            action=policy[state]
            transitions=transition(state,action)
            new_value=0
            for next_state,prob in transitions.items():
                r=reward(state,next_state)
                new_value+=prob*(r+gamma*V[next_state])
            V[state]=new_value
            delta=max(delta,abs(old_value-new_value))
        if delta<theta:
            break
    return V

Initial Policy
A->B
B->A
C->A
D->B
E->B
F->C
G->D


In [13]:
values=policy_evaluation(policy)

print("\n State Values \n")
for zone in zones:
    print(f"{zone}:{values[zone]:.2f}")


 State Values 

A:10.80
B:12.80
C:12.05
D:6.30
E:7.80
F:8.30
G:0.05


## Task C

Implement policy improvement to update the policy greedily basedon the value function.

In [16]:
def policy_improvement(V,gamma=0.0):
    new_policy={}
    policy_stable=True
    for state in zones:
        old_action=policy[state]
        best_action=None
        best_value=float("-inf")
        for action in actions[state]:
            action_value=0
            transitions=transition(state,action)
            for next_state,prob in transitions.items():
                r=reward(state,next_state)
                action_value+=prob*(r+gamma*V[next_state])
            if action_value>best_value:
                best_value=action_value
                best_action=action
        new_policy[state]=best_action
        if best_action!=old_action:
            policy_stable=False
    return new_policy,policy_stable

new_policy, stable = policy_improvement(values)

print("Improved Policy\n")

for zone in zones:
    print(f"{zone} -> {new_policy[zone]}")

print("\nPolicy Stable :", stable)

Improved Policy

A -> B
B -> A
C -> A
D -> B
E -> F
F -> C
G -> F

Policy Stable : False


## Task D

Iterate until the policy converges.

## Task E

Print the optimal policy (best action for each zone) and the valuefunction showing expected long‑term rewards.

In [17]:
def policy_iteration(gamma=0.9):
    policy = {}
    for zone in zones:
        policy[zone] = actions[zone][0]
    iteration = 1
    while True:

        print(f"\nIteration {iteration}")
        V = policy_evaluation(policy, gamma)
        globals()['policy'] = policy
        new_policy, stable = policy_improvement(V, gamma)
        print("Policy")
        for zone in zones:
            print(f"{zone} -> {new_policy[zone]}")
        if stable:
            print("\nPolicy Converged!")
            return new_policy, V
        policy = new_policy
        iteration += 1
optimal_policy, optimal_values = policy_iteration()
print("\nOptimal Policy\n")

for zone in zones:
    print(f"{zone} -> {optimal_policy[zone]}")

print("\nOptimal Value Function\n")

for zone in zones:
    print(f"{zone} : {optimal_values[zone]:.2f}")


Iteration 1
Policy
A -> B
B -> A
C -> A
D -> B
E -> B
F -> C
G -> F

Iteration 2
Policy
A -> B
B -> A
C -> A
D -> B
E -> B
F -> C
G -> F

Policy Converged!

Optimal Policy

A -> B
B -> A
C -> A
D -> B
E -> B
F -> C
G -> F

Optimal Value Function

A : 117.26
B : 118.74
C : 117.71
D : 111.10
E : 113.16
F : 112.95
G : 109.54


## Task F

Compare with value iteration algorithm and write your intuitions.

In [18]:
def value_iteration(gamma=0.9, theta=1e-6):
    V = {zone: 0 for zone in zones}
    while True:
        delta = 0
        for state in zones:
            old_value = V[state]
            best_value = float("-inf")
            for action in actions[state]:
                action_value = 0
                transitions = transition(state, action)
                for next_state, prob in transitions.items():
                    r = reward(state, next_state)
                    action_value += prob * (
                        r + gamma * V[next_state]
                    )
                best_value = max(best_value, action_value)
            V[state] = best_value
            delta = max(delta, abs(old_value - best_value))
        if delta < theta:
            break
    return V

def extract_policy(V, gamma=0.9):
    policy = {}
    for state in zones:
        best_action = None
        best_value = float("-inf")
        for action in actions[state]:
            action_value = 0
            transitions = transition(state, action)
            for next_state, prob in transitions.items():
                r = reward(state, next_state)
                action_value += prob * (
                    r + gamma * V[next_state]
                )
            if action_value > best_value:
                best_value = action_value
                best_action = action
        policy[state] = best_action
    return policy
V_vi = value_iteration()
policy_vi = extract_policy(V_vi)
print("Optimal Policy (Value Iteration)")

for zone in zones:
    print(f"{zone} -> {policy_vi[zone]}")
print("Optimal Value Function")

for zone in zones:
    print(f"{zone} : {V_vi[zone]:.2f}")

Optimal Policy (Value Iteration)
A -> B
B -> A
C -> A
D -> B
E -> B
F -> C
G -> F
Optimal Value Function
A : 117.26
B : 118.74
C : 117.71
D : 111.10
E : 113.16
F : 112.95
G : 109.54


## Inferences
Successfully represented the taxi fleet management problem as an MDP by defining the zones as states, possible movements as actions, transition probabilities based on traffic conditions, and rewards based on passenger availability and travel cost.

The value function showed the expected long term reward for each zone under the initial policy. Zones with higher passenger demand, such as A and B, obtained higher values, while low-demand zones like D and G had comparatively lower values.

The policy improvement step compared all possible actions from every zone and selected the action with the highest expected reward. Since the policy changed (Policy Stable = False), it indicated that the initial policy was not yet optimal.

After repeatedly performing policy evaluation and policy improvement, the policy converged in 2 iterations. This shows that the algorithm successfully found an optimal policy for the given taxi management problem.

The Value Iteration algorithm produced the same optimal policy and value function as Policy Iteration. This confirms that both algorithms reached the same optimal solution, although they follow different approaches to compute it.